# Redox FHIR Pipeline - Bronze Ingestion

This notebook ingests raw Redox FHIR bundles from a mounted location into Bronze streaming tables.

## Tables Created
| Table | Description |
|-------|-------------|
| `fhir_bronze` | Raw JSON as STRING with file metadata |
| `fhir_bronze_variant` | Parsed JSON as VARIANT for flexible querying |

## Architecture
```
Mount (JSON files) → Autoloader → fhir_bronze (STRING) → fhir_bronze_variant (VARIANT)
```

_Note: Attach to a Serverless SQL Warehouse for execution._

In [ ]:
-- ============================================================================
-- CONFIGURATION
-- ============================================================================
DECLARE OR REPLACE VARIABLE catalog_use STRING DEFAULT 'redox_fhir';
DECLARE OR REPLACE VARIABLE schema_use STRING DEFAULT 'bronze';
DECLARE OR REPLACE VARIABLE mount_path STRING DEFAULT '/mnt/redox/fhir/';

-- Override with job parameters if provided
SET VARIABLE catalog_use = COALESCE(:catalog_use, catalog_use);
SET VARIABLE schema_use = COALESCE(:schema_use, schema_use);
SET VARIABLE mount_path = COALESCE(:mount_path, mount_path);

USE IDENTIFIER(catalog_use || '.' || schema_use);
SELECT current_catalog() AS catalog, current_schema() AS schema, mount_path;

## Step 1: Create Bronze Table (Raw JSON as STRING)

This streaming table ingests FHIR bundles as raw text, preserving the original JSON and capturing file metadata for lineage.

In [ ]:
DECLARE OR REPLACE VARIABLE create_bronze_stmt STRING;

SET VARIABLE create_bronze_stmt = "
CREATE OR REFRESH STREAMING TABLE fhir_bronze (
  file_metadata STRUCT<
    file_path: STRING,
    file_name: STRING,
    file_size: BIGINT,
    file_block_start: BIGINT,
    file_block_length: BIGINT,
    file_modification_time: TIMESTAMP
  > NOT NULL COMMENT 'File metadata from source'
  ,ingest_time TIMESTAMP NOT NULL DEFAULT CURRENT_TIMESTAMP() COMMENT 'Ingestion timestamp'
  ,bundle_uuid STRING NOT NULL COMMENT 'Unique identifier for each FHIR bundle'
  ,value STRING COMMENT 'Raw JSON content as string'
)
COMMENT 'Raw Redox FHIR bundles ingested as text - Bronze layer'
TBLPROPERTIES (
  'delta.enableChangeDataFeed' = 'true'
  ,'delta.enableDeletionVectors' = 'true'
  ,'delta.enableRowTracking' = 'true'
  ,'quality' = 'bronze'
)
AS SELECT
  _metadata AS file_metadata
  ,CURRENT_TIMESTAMP() AS ingest_time
  ,uuid() AS bundle_uuid
  ,value
FROM STREAM read_files(
  '" || mount_path || "'
  ,format => 'text'
  ,wholeText => true
)
";

SELECT create_bronze_stmt AS statement;

In [ ]:
EXECUTE IMMEDIATE create_bronze_stmt;

In [ ]:
-- Verify bronze table
SELECT 
  bundle_uuid,
  file_metadata.file_name,
  file_metadata.file_size,
  ingest_time,
  LEFT(value, 200) AS json_preview
FROM fhir_bronze
ORDER BY ingest_time DESC
LIMIT 5;

## Step 2: Create Bronze Variant Table (Parsed JSON as VARIANT)

This streaming table parses the raw JSON into Databricks VARIANT type, enabling schema-agnostic querying without predefined schemas.

In [ ]:
CREATE OR REFRESH STREAMING TABLE fhir_bronze_variant (
  bundle_uuid STRING NOT NULL COMMENT 'Unique identifier for each FHIR bundle'
  ,ingest_time TIMESTAMP NOT NULL COMMENT 'Ingestion timestamp'
  ,file_metadata STRUCT<
    file_path: STRING,
    file_name: STRING,
    file_size: BIGINT,
    file_block_start: BIGINT,
    file_block_length: BIGINT,
    file_modification_time: TIMESTAMP
  > NOT NULL COMMENT 'File metadata from source'
  ,fhir VARIANT COMMENT 'Parsed FHIR bundle as VARIANT'
)
COMMENT 'Parsed Redox FHIR bundles as VARIANT - Bronze layer'
TBLPROPERTIES (
  'delta.enableChangeDataFeed' = 'true'
  ,'delta.enableDeletionVectors' = 'true'
  ,'delta.enableRowTracking' = 'true'
  ,'quality' = 'bronze'
  ,'pipelines.channel' = 'PREVIEW'
  ,'delta.feature.variantType-preview' = 'supported'
)
AS SELECT
  bundle_uuid
  ,ingest_time
  ,file_metadata
  ,try_parse_json(value) AS fhir
FROM STREAM fhir_bronze;

In [ ]:
-- Verify variant table and explore bundle structure
SELECT
  bundle_uuid,
  fhir:resourceType::STRING AS resource_type,
  fhir:type::STRING AS bundle_type,
  fhir:id::STRING AS bundle_id,
  fhir:timestamp::STRING AS bundle_timestamp,
  COALESCE(size(variant_get(fhir, '$.entry')::ARRAY<VARIANT>), 0) AS entry_count
FROM fhir_bronze_variant
ORDER BY ingest_time DESC
LIMIT 5;

In [ ]:
-- Check for Redox-specific Meta information
SELECT 
  bundle_uuid,
  fhir:Meta AS redox_meta,
  fhir:Meta.profile AS meta_profile
FROM fhir_bronze_variant
LIMIT 5;